# Session 24 — Pandas analysis
## Follow-along

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ffraile/computer_science_tutorials/blob/main/source/Modularization/follow_along/Session%2024%20-%20Pandas%20analysis%20-%20follow%20along.ipynb) [![Github](https://img.shields.io/badge/download-github-blue?logo=github)](https://github.com/ffraile/computer_science_tutorials/raw/main/source/Modularization/follow_along/Session%2024%20-%20Pandas%20analysis%20-%20follow%20along.ipynb)


## Learning goals

- define table grain, keys and expected cardinality;
- perform and compare `inner` and `left` joins;
- detect unmatched records and duplicated lookup keys;
- validate a many-to-one relationship;
- calculate derived values and sort records;
- group and aggregate by one or more variables;
- reconcile totals and visualise an aggregate result.


## Working routine

> **predict → execute → validate → interpret**

Before every join, state the expected relationship and row count. Before every aggregation, state the result grain and a total that should be preserved.


## Use of AI assistance

Copy the complete prompt below into a new Gemini conversation. Include your predicted grain, key, row count or total when requesting help. You remain responsible for verifying every suggestion.

```text
You are a coding tutor for an introductory Computer Science course. Help me reason and work independently; do not complete my work for me.

TUTORING RULES
- Begin by asking for my goal, prediction, current attempt or observed evidence.
- Ask one focused question at a time and wait for my answer.
- Use the lowest useful help level: clarify → one conceptual hint → steps or incomplete pseudocode → a small incomplete code fragment → review code I wrote.
- Never provide a complete exercise solution or fill an answer cell unless the teacher explicitly enables COMPLETE_SOLUTION.
- Preserve notebook statements, tests and existing code. Do not complete future cells.
- Follow predict → execute → validate → interpret.
- When debugging, use evidence, one hypothesis, one diagnostic check and the smallest correction.
- Keep each response concise and wait after one question or next step.

ACTIVITY CONTEXT
- Session: 24 — Pandas analysis
- Activity type: FOLLOW-ALONG
- Students already know: DataFrames and Series; CSV loading; inspection; loc and iloc; Boolean filters; missing values; type conversion; duplicates; justified cleaning; NumPy and Matplotlib
- This activity introduces: table grain; keys and cardinality; merge with inner and left joins; unmatched records; duplicate lookup keys; indicator and validate; derived columns; sort_values; groupby; count, sum, mean, min and max; named aggregation; total reconciliation; and simple aggregate visualisation
- Do not use: right or outer joins in depth; concatenation; joins on indexes; MultiIndex; pivot tables; apply(); lambda functions; rolling operations; advanced plotting; classes; recursion; generators; or decorators
- Maximum help: PARTIAL_CODE
```


## Scenario — Orders and product catalogue

The orders table records transactions. The product catalogue supplies names, categories and prices. We need sales by category without losing orders or multiplying them accidentally.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.DataFrame([
    {"order_id":"O-101", "product_id":"P-01", "quantity":2},
    {"order_id":"O-102", "product_id":"P-02", "quantity":1},
    {"order_id":"O-103", "product_id":"P-01", "quantity":3},
    {"order_id":"O-104", "product_id":"P-03", "quantity":4},
    {"order_id":"O-105", "product_id":"P-99", "quantity":1},
])

products = pd.DataFrame([
    {"product_id":"P-01", "product_name":"Sensor kit", "category":"Electronics", "unit_price":25.0},
    {"product_id":"P-02", "product_name":"Cable pack", "category":"Accessories", "unit_price":8.0},
    {"product_id":"P-03", "product_name":"Mounting set", "category":"Accessories", "unit_price":12.0},
])


## 1. Define grain, keys and cardinality

| Table | One row represents | Candidate key |
|---|---|---|
| `orders` | one order line | `order_id` |
| `products` | one catalogue product | `product_id` |

The intended relationship is many order lines to one product. Check uniqueness and missing keys before merging.


In [ ]:
print("Order rows:", len(orders))
print("Unique order IDs?", orders["order_id"].is_unique)
print("Unique catalogue product IDs?", products["product_id"].is_unique)
print("Missing order product IDs:", orders["product_id"].isna().sum())


## 2. Compare inner and left joins

Predict the row count and fate of `O-105` in each result.


In [ ]:
inner_result = orders.merge(products, on="product_id", how="inner")
left_result = orders.merge(products, on="product_id", how="left", indicator=True)

print("Inner rows:", len(inner_result))
print("Left rows:", len(left_result))
display(left_result)


## 3. Audit unmatched records

The merge indicator records `both`, `left_only` or `right_only`. In a left join, `left_only` identifies order records with no catalogue match.


In [ ]:
unmatched_orders = left_result.loc[left_result["_merge"] == "left_only"]
display(unmatched_orders)
print("Unmatched product IDs:", unmatched_orders["product_id"].unique())


## 4. Validate the relationship

Use `validate="many_to_one"` to turn the catalogue-uniqueness assumption into an executable check.


In [ ]:
enriched = orders.merge(
    products,
    on="product_id",
    how="left",
    indicator=True,
    validate="many_to_one",
)

assert len(enriched) == len(orders)


### Why duplicates are dangerous

If `product_id` appeared twice in the catalogue, matching order rows would multiply. The join might execute, but the result would no longer have one row per order line. `validate` prevents that silent change.


## 5. Separate matched records for sales analysis

The unknown product has no trusted price. Keep it in the audit output, but exclude it from a calculation requiring catalogue information.


In [ ]:
matched_orders = enriched.loc[enriched["_merge"] == "both"].copy()
matched_orders = matched_orders.drop(columns="_merge")
matched_orders["sales"] = matched_orders["quantity"] * matched_orders["unit_price"]

display(matched_orders)


## 6. Sort records

`sort_values()` orders rows without changing their values. Sort by sales descending and use `order_id` as a deterministic tie-breaker.


In [ ]:
ranked_orders = matched_orders.sort_values(
    ["sales", "order_id"],
    ascending=[False, True],
)
display(ranked_orders)


## 7. Group and aggregate

Grouping defines membership. Named aggregation defines clear output measures and column names. Predict one result row per category.


In [ ]:
category_summary = (
    matched_orders
    .groupby("category", as_index=False)
    .agg(
        order_lines=("order_id", "count"),
        total_units=("quantity", "sum"),
        total_sales=("sales", "sum"),
        mean_unit_price=("unit_price", "mean"),
        min_unit_price=("unit_price", "min"),
        max_unit_price=("unit_price", "max"),
    )
    .sort_values("total_sales", ascending=False)
)

display(category_summary)


## 8. Reconcile totals

The sum of category sales must equal the sum of included order-line sales. Counts and units can be reconciled in the same way.


In [ ]:
assert category_summary["total_sales"].sum() == matched_orders["sales"].sum()
assert category_summary["total_units"].sum() == matched_orders["quantity"].sum()
assert category_summary["order_lines"].sum() == len(matched_orders)

print("Reconciled sales:", category_summary["total_sales"].sum())


## 9. Group by two variables

Adding a second grouping field changes the result grain to one row per category and product.


In [ ]:
product_summary = (
    matched_orders
    .groupby(["category", "product_name"], as_index=False)
    .agg(total_units=("quantity", "sum"), total_sales=("sales", "sum"))
)
display(product_summary)


## 10. Visualise and interpret

The chart communicates one chosen aggregation. It does not show unmatched orders and does not validate the join.


In [ ]:
category_summary.plot.bar(x="category", y="total_sales", legend=False)
plt.ylabel("Sales")
plt.title("Sales by product category")
plt.tight_layout()
plt.show()


## Final checkpoint

1. What does one row represent before and after the join?
2. Why was a left join useful for auditing?
3. What does `validate="many_to_one"` protect?
4. Why was the unknown product excluded from sales but retained in the audit?
5. What is the result grain of each summary?
6. Which reconciliation checks demonstrate that aggregation preserved included records?
